# OrthoFinder
GitHub: [OrthoFinder/OrthoFinder](https://github.com/OrthoFinder/OrthoFinder)  
Website: [orthofinder.github.io](https://orthofinder.github.io/OrthoFinder/)


## Section jump list

- **[1. What is OrthoFinder?](#sec-1)**
- **[2. How to install OrthoFinder?](#sec-2)**
- **[3. Run ExampleData from source code](#sec-3)**
  - [3.1 Check environment, source code, and dependency status](#sec-3-1)
  - [3.2 Environment setup](#sec-3-2)
  - [3.3 Download the source code](#sec-3-3)
  - [3.4 Install dependencies](#sec-3-4)
  - [3.5 Test the installation](#sec-3-5)
  - [3.6 Run ExampleData (explanation)](#sec-3-6)
  - [3.7 Directory structure](#sec-3-7)
  - [3.8 Main run](#sec-3-8)
- **[4. Check the results directory](#sec-4)**
- **[5. Review the ExampleData results](#sec-5)**
  - [5.1 Basic information](#sec-5-1)
  - [5.2 Quality control](#sec-5-2)
  - [5.3.1 Homologs/orthologs/paralogs/orthogroups](#sec-5-3-1)
  - [5.3.2 Gene trees](#sec-5-3-2)
  - [5.3.3 Gene duplications](#sec-5-3-3)
- **[6. Core/Assign](#sec-6)**

<a id="sec-1"></a>
## 1. What is OrthoFinder?

OrthoFinder identifies orthogroups, infers gene trees for all orthogroups, and analyzes these gene trees to infer a rooted species tree. It then identifies gene duplication events across the complete set of gene trees and analyzes them at both the gene-tree and species-tree levels. OrthoFinder uses this phylogenetic information to identify the complete set of orthologs between species and provides extensive comparative genomics statistics.


![image](https://raw.githubusercontent.com/OrthoFinder/OrthoFinder/gh-pages/assets/images/workflow.png)

- **OrthoFinder 2**

  ```bash
  orthofinder -f ExampleData
  ```

- **OrthoFinder 3**

  Run the core analysis:

  ```bash
  orthofinder -f ExampleData
  ```

  Assign additional species to an existing core analysis:

  ```bash
  orthofinder --core ExampleData/OrthoFinder/Results_Sep03 --assign ExampleData/AdditionalSpecies
  ```


<a id="sec-2"></a>
## 2. How to install OrthoFinder?

**Environment: Linux**

### 2.1 Via Conda (recommended)

```bash
conda create -n of3_env python=3.12
conda activate of3_env
conda install orthofinder
```

### 2.2 Via GitHub

```bash
python3 -m venv of3_env
. of3_env/bin/activate
pip install git+https://github.com/OrthoFinder/OrthoFinder.git
```

### 2.3 From source code

```bash
# Download using Git
git clone https://github.com/OrthoFinder/OrthoFinder.git

# Alternatively, on a compatible Linux Intel machine, download and extract
# the OrthoFinder release archive
mkdir OrthoFinder && \
  wget -qO- https://github.com/OrthoFinder/OrthoFinder/releases/download/v3.1.5/orthofinder-linux-intel-3.1.5.tar.gz | \
  tar -xz --strip-components=1 -C OrthoFinder

cd OrthoFinder
python3 -m venv of3_env      # Create a virtual environment named of3_env
. of3_env/bin/activate       # Activate of3_env
pip install .
```


<a id="sec-3"></a>
## 3. Run ExampleData from source code

This section demonstrates a complete OrthoFinder run using the bundled `ExampleData` dataset. We will create an isolated Python environment, obtain the OrthoFinder source code, install the required Python packages, confirm that OrthoFinder can be launched, and then run the example analysis.

<a id="sec-3-1"></a>
### 3.1 Check environment, source code, and dependency status

Before doing any setup, check what's already in place. This tutorial does not 
install OrthoFinder for you — it only reports whether the virtual environment,
the OrthoFinder source code, and its Python dependencies are ready. Use the
result to decide which of the setup cells below you actually need to run.

If the summary below reports that everything is ready, you can skip sections
3.2–3.4 entirely and go straight to running the analysis. Otherwise, run
sections 3.2, 3.3, and 3.4 in order — each one checks its own prerequisite
and skips itself if it's already satisfied, so it's safe to run all three
even if only one of them is actually missing.

In [2]:
%%bash
ENV_DIR="of3_env"
SRC_DIR="OrthoFinder"
REQS_FILE="$SRC_DIR/requirements.txt"

# 1. Virtual environment
if [ -x "$ENV_DIR/bin/python" ]; then
    ENV_READY=1
    echo "[OK]      Virtual environment '$ENV_DIR' is ready."
else
    ENV_READY=0
    echo "[MISSING] Virtual environment '$ENV_DIR' was not found."
fi

# 2. OrthoFinder source code
if [ -d "$SRC_DIR" ] && [ -f "$SRC_DIR/orthofinder.py" ]; then
    SRC_READY=1
    echo "[OK]      OrthoFinder source code is ready in '$SRC_DIR'."
else
    SRC_READY=0
    echo "[MISSING] OrthoFinder source code was not found in '$SRC_DIR'."
fi

# 3. Python dependencies (only checkable once the venv and requirements.txt both exist)
if [ "$ENV_READY" -eq 1 ] && [ -f "$REQS_FILE" ]; then
    if "$ENV_DIR/bin/python" -c "
import re, sys
from importlib import metadata

missing = []
with open('$REQS_FILE') as f:
    for line in f:
        line = line.split('#', 1)[0].strip()
        if not line:
            continue
        name = re.split(r'[<>=~!\[]', line, 1)[0].strip()
        try:
            metadata.version(name)
        except metadata.PackageNotFoundError:
            missing.append(name)
sys.exit(1 if missing else 0)
"; then
        DEPS_READY=1
        echo "[OK]      All dependencies in '$REQS_FILE' are installed."
    else
        DEPS_READY=0
        echo "[MISSING] Some dependencies in '$REQS_FILE' are not installed."
    fi
else
    DEPS_READY=0
    echo "[SKIPPED] Dependency check requires both the virtual environment and '$REQS_FILE'."
fi

echo ""
if [ "$ENV_READY" -eq 1 ] && [ "$SRC_READY" -eq 1 ] && [ "$DEPS_READY" -eq 1 ]; then
    echo "Everything is ready — you can skip ahead and run the OrthoFinder analysis."
else
    echo "Some prerequisites are missing — run the relevant setup cells below before proceeding."
fi

[OK]      Virtual environment 'of3_env' is ready.
[OK]      OrthoFinder source code is ready in 'OrthoFinder'.
[OK]      All dependencies in 'OrthoFinder/requirements.txt' are installed.

Everything is ready — you can skip ahead and run the OrthoFinder analysis.


<a id="sec-3-2"></a>
### 3.2 Environment setup

Create a dedicated Python virtual environment for this tutorial. Using a separate environment keeps the OrthoFinder dependencies isolated from other Python packages installed on your system.


In [ ]:
%%bash
set -e

python3 -m venv of3_env
source of3_env/bin/activate

# Install the packages required for this notebook without displaying pip output.
python -m pip install --upgrade pip -q >/dev/null 2>&1
python -m pip install ipykernel pandas -q >/dev/null 2>&1

echo "Python environment and notebook dependencies installed successfully."


<a id="sec-3-3"></a>
### 3.3 Download the source code

Download the OrthoFinder source code into the current working directory. The cell below first checks whether an `OrthoFinder` directory already exists. If it does, the download is skipped so that an existing copy is not overwritten.


In [ ]:
%%bash
set -e
source of3_env/bin/activate

REPO_URL="https://github.com/OrthoFinder/OrthoFinder.git"
BRANCH="july-fix"
TARBALL_URL="https://github.com/OrthoFinder/OrthoFinder/releases/download/v3.1.5/orthofinder-linux-intel-3.1.5.tar.gz"

if [ -d "OrthoFinder" ]; then
    echo "OrthoFinder source code is already available; download skipped."
elif command -v git &> /dev/null; then
    git clone -q -b "$BRANCH" "$REPO_URL" OrthoFinder >/dev/null 2>&1
    echo "OrthoFinder source code downloaded successfully with Git."
elif command -v wget &> /dev/null; then
    mkdir -p OrthoFinder
    wget -qO- "$TARBALL_URL" | tar -xz --strip-components=1 -C OrthoFinder
    echo "OrthoFinder source code downloaded successfully with wget."
elif command -v curl &> /dev/null; then
    mkdir -p OrthoFinder
    curl -fsSL "$TARBALL_URL" | tar -xz --strip-components=1 -C OrthoFinder
    echo "OrthoFinder source code downloaded successfully with curl."
else
    python - <<'PY'
import io
import tarfile
import urllib.request

url = "https://github.com/OrthoFinder/OrthoFinder/releases/download/v3.1.5/orthofinder-linux-intel-3.1.5.tar.gz"

with urllib.request.urlopen(url) as response:
    data = response.read()

with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as archive:
    members = archive.getmembers()
    for member in members:
        parts = member.name.split("/", 1)
        if len(parts) == 2:
            member.name = parts[1]
            if member.name:
                archive.extract(member, "OrthoFinder", filter="data")
PY
    echo "OrthoFinder source code downloaded successfully with Python."
fi


<a id="sec-3-4"></a>
### 3.4 Install dependencies

Install the Python packages required by OrthoFinder into the virtual environment created above. The installation output is suppressed to keep the notebook concise. If the command completes successfully, a confirmation message is displayed.


In [ ]:
%%bash
set -e
source of3_env/bin/activate

python -m pip install -r OrthoFinder/requirements.txt -q >/dev/null 2>&1

echo "OrthoFinder dependencies installed successfully."


<a id="sec-3-5"></a>
### 3.5 Test the installation

Before running an analysis, confirm that OrthoFinder can be launched from the virtual environment. The command below displays the OrthoFinder help message; seeing the command-line options indicates that the installation is working correctly.


In [4]:
%%bash
cd OrthoFinder
../of3_env/bin/python -m orthofinder --help

/home/biol0178/qfo/OrthoUniversity/OrthoFinder/src/orthofinder/tools/tree.py:371: SyntaxWarning: invalid escape sequence '\-'
  """
/home/biol0178/qfo/OrthoUniversity/OrthoFinder/src/orthofinder/tools/newick.py:341: SyntaxWarning: invalid escape sequence '\s'
  MATCH = '%s\s*%s\s*(%s)?' % (FIRST_MATCH, SECOND_MATCH, _NHX_RE)



SIMPLE USAGE:
 Run full OrthoFinder analysis on FASTA format proteomes in <dir>
   orthofinder [options] -f <dir>

 To assign species from <dir1> to existing OrthoFinder orthogroups in <dir2>
   orthofinder [options] --assign <dir1> --core <dir2>

OPTIONS:
 -t <int>                Number of parallel sequence search threads [Default =  
                         24]                                                    
 -a <int>                Number of parallel analysis threads                    
 -M <txt>                Method for gene tree inference. Options "dendroblast"  
                         & "msa" [Default = msa]                                
 -S <txt>                Sequence search program [Default = diamond]            
                         Options: diamond, diamond_ultra_sens, blastp, mmseqs,  
                         blastn                                                 
 -A <txt>                MSA program, requires "-M msa" [Default = famsa]       
             

<a id="sec-3-6"></a>
### 3.6 Run ExampleData

There are three ways to run OrthoFinder on the `ExampleData` dataset:

- **Run OrthoFinder from an installed environment**

  ```bash
  orthofinder -f ExampleData
  ```

  This is the preferred method if OrthoFinder has been installed via Conda or is otherwise available in your current environment.

- **Run OrthoFinder directly with `orthofinder.py`**

  ```bash
  python orthofinder.py -f ExampleData
  ```

  This method can be used without installing OrthoFinder.

- **Run OrthoFinder as a Python module**

  ```bash
  python -m orthofinder -f ExampleData
  ```

  This also allows OrthoFinder to be run without a separate installation step.


<a id="sec-3-7"></a>
### 3.7 Directory structure

Before running OrthoFinder, inspect the current directory structure. At this point, the working directory should contain the `of3_env` virtual environment and the downloaded `OrthoFinder` source directory.


In [3]:
import os

def print_tree(path=".", prefix="", depth=0, max_depth=None, filelimit=None):
    """Pure-Python replacement for the `tree` command (which isn't
    guaranteed to be installed in every Jupyter/JupyterHub environment)."""
    try:
        entries = sorted(os.listdir(path))
    except (PermissionError, FileNotFoundError):
        return

    if filelimit is not None and len(entries) > filelimit:
        print(f"{prefix}[{len(entries)} entries exceeds filelimit, not opening dir]")
        return

    for i, entry in enumerate(entries):
        full_path = os.path.join(path, entry)
        is_last = (i == len(entries) - 1)
        connector = "\u2514\u2500\u2500 " if is_last else "\u251c\u2500\u2500 "
        print(f"{prefix}{connector}{entry}")
        if os.path.isdir(full_path) and (max_depth is None or depth + 1 < max_depth):
            extension = "    " if is_last else "\u2502   "
            print_tree(full_path, prefix + extension, depth + 1, max_depth, filelimit)

def show_tree(path=".", max_depth=None, filelimit=None):
    print(os.path.basename(os.path.abspath(path)) + "/")
    print_tree(path, max_depth=max_depth, filelimit=filelimit)

show_tree(".", max_depth=2)


6-OrthoFinder/
├── .ipynb_checkpoints
│   ├── README-checkpoint.txt
│   └── orthofinder_demonstration-checkpoint.ipynb
├── OrthoFinder
│   ├── .git
│   ├── .github
│   ├── .gitignore
│   ├── CONTRIBUTING.md
│   ├── ExampleData
│   ├── LICENSE
│   ├── MANIFEST.in
│   ├── Makefile
│   ├── README.md
│   ├── __pycache__
│   ├── assets
│   ├── orthofinder.py
│   ├── pyproject.toml
│   ├── requirements.txt
│   ├── requirements_dev.txt
│   ├── setup.py
│   ├── src
│   ├── tests
│   ├── tools
│   └── user_config.json
├── of3_env
│   ├── bin
│   ├── include
│   ├── lib
│   ├── lib64
│   ├── pyvenv.cfg
│   └── share
├── orthofinder-linux-intel-3.1.5.tar.gz
└── orthofinder_demonstration.ipynb


<a id="sec-3-8"></a>
### 3.8 Main run

In [4]:
%%bash
cd OrthoFinder
rm -rf ExampleData/OrthoFinder # cleanup the results dir
../of3_env/bin/python -m orthofinder -f ExampleData -pof  # Run OrthoFinder with -pof to produce the pairwise ortholog files


2026-09-02 16:01:51 : Starting OrthoFinder v3.1.5.post1.dev8
32 thread(s) for highly parallel tasks (BLAST searches etc.)
4 thread(s) for OrthoFinder algorithm

OrthoFinder version 3.1.5.post1.dev8 Copyright (C) 2014 David Emms

Results directory:
    /home/jovyan/6-OrthoFinder/OrthoFinder/ExampleData/OrthoFinder/Results_Sep02
/

Checking required programs are installed
Running with the recommended MSA tree inference by default. To revert to legacy 
method use "-M dendroblast".

Test can run "mcl" - ok
Test can run "famsa" - ok
Test can run "fasttree" - ok

Dividing up work for BLAST for parallel processing
--------------------------------------------------
Processing... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   4/4 0:00:000:00:00

Running diamond all-versus-all
Using 32 thread(s)
2026-09-02 16:01:51 : This may take some time...
Processing... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   16/16 0:00:03m15/16 0:00:03
2026-09-02 16:01:55 : Done all-versus-all sequence search

Running Ortho

<a id="sec-4"></a>
## 4. Check the OrthoFinder results directory

After the ExampleData analysis finishes, OrthoFinder creates its output under `ExampleData/OrthoFinder/`. The following cells inspect the generated directory structure and the `WorkingDirectory`, which contains intermediate files used during the analysis.

The exact results-directory name contains a date and may therefore differ between runs.


In [5]:
RESULTS_DIR = "OrthoFinder/ExampleData"
print(f"Found results directory: {RESULTS_DIR}")
show_tree(RESULTS_DIR, max_depth=3, filelimit=20)


Found results directory: OrthoFinder/ExampleData
ExampleData/
├── AdditionalSpecies
│   ├── M_arthritidis.fa
│   └── M_haemocanis.fa
├── Mycoplasma_agalactiae.faa
├── Mycoplasma_gallisepticum.faa
├── Mycoplasma_genitalium.faa
├── Mycoplasma_hyopneumoniae.faa
└── OrthoFinder
    └── Results_Sep02
        ├── Citation.txt
        ├── Comparative_Genomics_Statistics
        ├── Gene_Duplication_Events
        ├── Log.txt
        ├── MultipleSequenceAlignments
        ├── Orthogroup_Sequences
        ├── Orthogroups
        ├── Orthologues
        ├── Phylogenetic_Hierarchical_Orthogroups
        ├── Phylogenetically_Misplaced_Genes
        ├── Putative_Xenologs
        ├── Resolved_Gene_Trees
        ├── Single_Copy_Orthologue_Sequences
        ├── Species_Tree
        └── WorkingDirectory


In [6]:
%%bash
RESULTS_DIR=$(ls -d OrthoFinder/ExampleData/OrthoFinder/Results_*/ | tail -n 1)
echo "Found results directory: $RESULTS_DIR"
ls -al "$RESULTS_DIR"/WorkingDirectory

Found results directory: OrthoFinder/ExampleData/OrthoFinder/Results_Sep02/
total 3108
drwxr-sr-x  7 jovyan users   4096 Sep  2 16:02 .
drwxr-sr-x 15 jovyan users   4096 Sep  2 16:02 ..
drwxr-sr-x  2 jovyan users  20480 Sep  2 16:01 Alignments_ids
-rw-r--r--  1 jovyan users  42559 Sep  2 16:01 Blast0_0.txt.gz
-rw-r--r--  1 jovyan users  20941 Sep  2 16:01 Blast0_1.txt.gz
-rw-r--r--  1 jovyan users  16335 Sep  2 16:01 Blast0_2.txt.gz
-rw-r--r--  1 jovyan users  23424 Sep  2 16:01 Blast0_3.txt.gz
-rw-r--r--  1 jovyan users  21010 Sep  2 16:01 Blast1_0.txt.gz
-rw-r--r--  1 jovyan users  54424 Sep  2 16:01 Blast1_1.txt.gz
-rw-r--r--  1 jovyan users  19239 Sep  2 16:01 Blast1_2.txt.gz
-rw-r--r--  1 jovyan users  22000 Sep  2 16:01 Blast1_3.txt.gz
-rw-r--r--  1 jovyan users  16035 Sep  2 16:01 Blast2_0.txt.gz
-rw-r--r--  1 jovyan users  19142 Sep  2 16:01 Blast2_1.txt.gz
-rw-r--r--  1 jovyan users  15757 Sep  2 16:01 Blast2_2.txt.gz
-rw-r--r--  1 jovyan users  16936 Sep  2 16:01 Blast2_3.txt

<a id="sec-5"></a>
## 5. Review the ExampleData results

<a id="sec-5-1"></a>
### 5.1 Basic information

The `ExampleData` dataset contains four *Mycoplasma* species. OrthoFinder uses the protein sequences from these species to infer orthogroups, gene trees, ortholog relationships, gene duplication events, and a rooted species tree.

The species tree inferred for this example is shown below. The tips represent the four species in the analysis, while the internal nodes represent their inferred common ancestors.

![image](https://raw.githubusercontent.com/OrthoFinder/OrthoFinder/gh-pages/assets/images/species_tree.png)

The tree provides the phylogenetic framework used by OrthoFinder when interpreting gene trees. For example, it is used to distinguish speciation events from gene duplication events and to define hierarchical orthogroups at different ancestral nodes.

In the following sections, we will first perform a few basic quality-control checks and then explore some of the main OrthoFinder output files.

<a id="sec-5-2"></a>
### 5.2 Quality control

Before exploring the orthogroups, it is useful to check the quality of the OrthoFinder run. We want to confirm that most genes across all species have been assigned to orthogroups and that the inferred species tree is biologically plausible.

Open `Statistics_Overall.tsv` in the `Comparative_Genomics_Statistics` directory. This file summarizes the orthogroup assignment statistics across the complete dataset and can be viewed in spreadsheet software such as Microsoft Excel or in a text editor.

On the fifth line, the percentage of genes assigned to orthogroups is reported. In this example, the value is `81.0`.


In [7]:
%%bash
RESULTS_DIR=$(ls -d OrthoFinder/ExampleData/OrthoFinder/Results_*/ | tail -n 1)
echo "Found results directory: $RESULTS_DIR"
ls -al "$RESULTS_DIR"/Comparative_Genomics_Statistics

Found results directory: OrthoFinder/ExampleData/OrthoFinder/Results_Sep02/
total 48
drwxr-sr-x  2 jovyan users 4096 Sep  2 16:02 .
drwxr-sr-x 15 jovyan users 4096 Sep  2 16:02 ..
-rw-r--r--  1 jovyan users 5050 Sep  2 16:02 Duplications_per_Orthogroup.tsv
-rw-r--r--  1 jovyan users  209 Sep  2 16:02 Duplications_per_Species_Tree_Node.tsv
-rw-r--r--  1 jovyan users  261 Sep  2 16:02 OrthologuesStats_many-to-many.tsv
-rw-r--r--  1 jovyan users  269 Sep  2 16:02 OrthologuesStats_many-to-one.tsv
-rw-r--r--  1 jovyan users  266 Sep  2 16:02 OrthologuesStats_one-to-many.tsv
-rw-r--r--  1 jovyan users  282 Sep  2 16:02 OrthologuesStats_one-to-one.tsv
-rw-r--r--  1 jovyan users  282 Sep  2 16:02 OrthologuesStats_Totals.tsv
-rw-r--r--  1 jovyan users 1469 Sep  2 16:02 Statistics_Overall.tsv
-rw-r--r--  1 jovyan users 2654 Sep  2 16:02 Statistics_PerSpecies.tsv


In [8]:
%%bash
RESULTS_DIR=$(ls -d OrthoFinder/ExampleData/OrthoFinder/Results_*/ | tail -n 1)
echo "Found results directory: $RESULTS_DIR"
head -n 7 "$RESULTS_DIR"/Comparative_Genomics_Statistics/Statistics_Overall.tsv

Found results directory: OrthoFinder/ExampleData/OrthoFinder/Results_Sep02/
Number of species	4
Number of genes	2733
Number of genes in orthogroups	2215
Number of unassigned genes	518
Percentage of genes in orthogroups	81.0
Percentage of unassigned genes	19.0
Number of orthogroups	599


A useful rule of thumb is that more than 80% of genes should be assigned to orthogroups. A substantially lower value may indicate that some orthology relationships are being missed. In such cases, improved species sampling can often help.

Next, open `Statistics_PerSpecies.tsv` from the same directory. This file reports the percentage of genes assigned to orthogroups for each species individually, rather than across the complete dataset.

In this example, most genes are assigned to orthogroups across all species.


In [9]:
import pandas as pd
import glob

RESULTS_DIR_CORE = glob.glob("OrthoFinder/ExampleData/OrthoFinder/Results_*")[0]

df = pd.read_csv(
    f"{RESULTS_DIR_CORE}/Comparative_Genomics_Statistics/Statistics_PerSpecies.tsv",
    sep="\t",
)
with pd.option_context("display.max_colwidth", None):
    display(df.head(5))


,Unnamed: 0,Mycoplasma_agalactiae,Mycoplasma_gallisepticum,Mycoplasma_genitalium,Mycoplasma_hyopneumoniae
0,Number of genes,820,763,476,674
1,Number of genes in orthogroups,650,596,417,552
2,Number of unassigned genes,170,167,59,122
3,Percentage of genes in orthogroups,79.3,78.1,87.6,81.9
4,Percentage of unassigned genes,20.7,21.9,12.4,18.1


The lowest percentage is for `Mycoplasma_gallisepticum`, but 78.1% of its genes are still assigned to orthogroups. The key point is that it is always worth checking these statistics before interpreting the results. If one species has a particularly low assignment rate, adding more species may help reduce long evolutionary distances and improve orthology inference.

Another useful quality-control step is to inspect the species tree. You can open the tree in [iTOL](https://itol.embl.de/upload.cgi) either by copying and pasting the tree contents or by uploading the tree file directly.


In [10]:
%%bash
RESULTS_DIR=$(ls -d OrthoFinder/ExampleData/OrthoFinder/Results_*/ | tail -n 1)
echo "Found results directory: $RESULTS_DIR"
cat "$RESULTS_DIR/Species_Tree/SpeciesTree_rooted.txt"

Found results directory: OrthoFinder/ExampleData/OrthoFinder/Results_Sep02/
((Mycoplasma_agalactiae:0.51407,Mycoplasma_hyopneumoniae:0.53945)1:0.25523,(Mycoplasma_genitalium:0.53447,Mycoplasma_gallisepticum:0.4496)1:0.25523);

### 5.3 Interpreting the results

<a id="sec-5-3-1"></a>
#### 5.3.1 Homologs, orthologs, paralogs, and orthogroups

Homologous genes descend from a gene present in a common ancestor. Their evolutionary relationships can be classified according to the evolutionary event that separates their lineages in the gene tree.

<img src="https://www.nlm.nih.gov/ncbi/workshops/2023-08_BLAST_evol/img/ortholog_paralog.jpg" width="700">

*Figure adapted from the National Library of Medicine/NCBI tutorial on orthologs and paralogs.*  
Source: [NLM/NCBI — Homology: Orthologs and Paralogs](https://www.nlm.nih.gov/ncbi/workshops/2023-08_BLAST_evol/ortho_para.html)

- **Orthologs**

  Genes separated by a **speciation event** are called orthologs.

  In the figure above, the ancestral gene duplicated to produce `Gene 1a` and `Gene 1b`, and the ancestral species subsequently split into two descendant species. `Gene 1a` in Species 1 and `Gene 1a` in Species 2 are orthologs because their lineages were separated by the speciation event. The same applies to the two copies of `Gene 1b`.

- **Paralogs**

  Genes separated by a **gene duplication event** are called paralogs.

  In the same example, `Gene 1a` and `Gene 1b` are paralogs because they arose from a duplication of the ancestral `Gene 1`.

  Importantly, paralogs do **not** have to occur in the same species. For example, `Gene 1a` in Species 1 and `Gene 1b` in Species 2 are also paralogs because their lineages trace back to the ancestral duplication event.

- **Homologs**

  Orthologs and paralogs are both types of homologs. Therefore, all four genes shown in the figure are homologous because they descend from the same ancestral gene.

- **Orthogroups**

  An orthogroup is the set of genes descended from a single gene in the last common ancestor of the species being considered. In other words, an orthogroup contains the genes that trace back to one ancestral gene at the relevant phylogenetic level.

  OrthoFinder identifies orthogroups and then uses gene trees together with the species tree to infer ortholog relationships, gene duplication events, and hierarchical orthogroups at different phylogenetic levels.

  Hierarchical orthogroups can be defined at different nodes of the species tree, allowing orthogroup membership to be examined at different evolutionary depths.

> **Note:** Orthogroups are initially constructed using sequence-similarity information. OrthoFinder subsequently uses gene trees and the species tree to interpret the evolutionary relationships among genes and to identify phylogenetically informed hierarchical orthogroups.

After orthogroup inference, OrthoFinder infers a gene tree for each orthogroup and reconciles it with the species tree to identify duplication events. If a duplication predates the relevant species-tree node, the gene tree can be split so that the resulting groups are consistent with the phylogenetic definition of an orthogroup. Applying this procedure across the species tree produces hierarchical orthogroups.

In short, orthogroup membership is not interpreted solely from sequence-similarity clustering; gene trees and the species tree are also used to provide a phylogenetic interpretation.

OrthoFinder writes ortholog relationships explicitly. Paralogs can be inferred as genes within the same orthogroup that are not orthologs of one another. Writing every paralogous relationship explicitly could require a large amount of disk space.

In [11]:
%%bash
RESULTS_DIR=$(ls -d OrthoFinder/ExampleData/OrthoFinder/Results_*/ | tail -n 1)
echo "Found results directory: $RESULTS_DIR"
ls -al "$RESULTS_DIR"/Orthogroups

Found results directory: OrthoFinder/ExampleData/OrthoFinder/Results_Sep02/
total 208
drwxr-sr-x  2 jovyan users  4096 Sep  2 16:02 .
drwxr-sr-x 15 jovyan users  4096 Sep  2 16:02 ..
-rw-r--r--  1 jovyan users 12707 Sep  2 16:02 Orthogroups.GeneCount.tsv
-rw-r--r--  1 jovyan users  2499 Sep  2 16:02 Orthogroups_SingleCopyOrthologues.txt
-rw-r--r--  1 jovyan users 69156 Sep  2 16:02 Orthogroups.tsv
-rw-r--r--  1 jovyan users 88094 Sep  2 16:02 Orthogroups.txt
-rw-r--r--  1 jovyan users 21825 Sep  2 16:02 Orthogroups_UnassignedGenes.tsv


In [12]:
# RESULTS_DIR_CORE was defined above and is reused in the following cells.


In [13]:
df = pd.read_csv(f"{RESULTS_DIR_CORE}/Orthogroups/Orthogroups.tsv", sep="\t")
with pd.option_context('display.max_colwidth', None):
    display(df.tail(5))

,Orthogroup,Mycoplasma_agalactiae,Mycoplasma_gallisepticum,Mycoplasma_genitalium,Mycoplasma_hyopneumoniae
594,OG0000594,NaN,NaN,NaN,"gi|71851864|gb|AAZ44472.1|, gi|71851869|gb|AAZ44477.1|"
595,OG0000595,NaN,NaN,NaN,"gi|71851867|gb|AAZ44475.1|, gi|71851872|gb|AAZ44480.1|"
596,OG0000596,NaN,NaN,NaN,"gi|144227700|gb|AAZ44653.2|, gi|71852048|gb|AAZ44656.1|"
597,OG0000597,NaN,NaN,NaN,"gi|144227730|gb|AAZ44708.2|, gi|144227731|gb|AAZ44711.2|"
598,OG0000598,NaN,NaN,NaN,"gi|144227732|gb|AAZ44712.2|, gi|71852101|gb|AAZ44709.1|"


In [14]:
%%bash
RESULTS_DIR=$(ls -d OrthoFinder/ExampleData/OrthoFinder/Results_*/ | tail -n 1)
echo "Found results directory: $RESULTS_DIR"
ls -al "$RESULTS_DIR"/Phylogenetic_Hierarchical_Orthogroups

Found results directory: OrthoFinder/ExampleData/OrthoFinder/Results_Sep02/
total 88
drwxr-sr-x  2 jovyan users  4096 Sep  2 16:02 .
drwxr-sr-x 15 jovyan users  4096 Sep  2 16:02 ..
-rw-r--r--  1 jovyan users 43582 Sep  2 16:02 N1.tsv
-rw-r--r--  1 jovyan users 35585 Sep  2 16:02 N2.tsv


In [15]:
df = pd.read_csv(f"{RESULTS_DIR_CORE}/Phylogenetic_Hierarchical_Orthogroups/N1.tsv", sep="\t")
with pd.option_context('display.max_colwidth', None):
    display(df.tail(5))

,HOG,OG,Gene Tree Parent Clade,Mycoplasma_agalactiae,Mycoplasma_gallisepticum,Mycoplasma_genitalium,Mycoplasma_hyopneumoniae
431,N1.HOG0000431,OG0000594,-,NaN,NaN,NaN,"gi|71851864|gb|AAZ44472.1|, gi|71851869|gb|AAZ44477.1|"
432,N1.HOG0000432,OG0000595,-,NaN,NaN,NaN,"gi|71851872|gb|AAZ44480.1|, gi|71851867|gb|AAZ44475.1|"
433,N1.HOG0000433,OG0000596,-,NaN,NaN,NaN,"gi|144227700|gb|AAZ44653.2|, gi|71852048|gb|AAZ44656.1|"
434,N1.HOG0000434,OG0000597,-,NaN,NaN,NaN,"gi|144227731|gb|AAZ44711.2|, gi|144227730|gb|AAZ44708.2|"
435,N1.HOG0000435,OG0000598,-,NaN,NaN,NaN,"gi|71852101|gb|AAZ44709.1|, gi|144227732|gb|AAZ44712.2|"


In [16]:
import glob

RESULTS_DIR = glob.glob("OrthoFinder/ExampleData/OrthoFinder/Results_*")[0]
print(f"Found results directory: {RESULTS_DIR}")
show_tree(f"{RESULTS_DIR}/Orthologues")


Found results directory: OrthoFinder/ExampleData/OrthoFinder/Results_Sep02
Orthologues/
├── Mycoplasma_agalactiae.tsv
├── Mycoplasma_gallisepticum.tsv
├── Mycoplasma_genitalium.tsv
├── Mycoplasma_hyopneumoniae.tsv
├── Orthologues_Mycoplasma_agalactiae
│   ├── Mycoplasma_agalactiae__v__Mycoplasma_gallisepticum.tsv
│   ├── Mycoplasma_agalactiae__v__Mycoplasma_genitalium.tsv
│   └── Mycoplasma_agalactiae__v__Mycoplasma_hyopneumoniae.tsv
├── Orthologues_Mycoplasma_gallisepticum
│   ├── Mycoplasma_gallisepticum__v__Mycoplasma_agalactiae.tsv
│   ├── Mycoplasma_gallisepticum__v__Mycoplasma_genitalium.tsv
│   └── Mycoplasma_gallisepticum__v__Mycoplasma_hyopneumoniae.tsv
├── Orthologues_Mycoplasma_genitalium
│   ├── Mycoplasma_genitalium__v__Mycoplasma_agalactiae.tsv
│   ├── Mycoplasma_genitalium__v__Mycoplasma_gallisepticum.tsv
│   └── Mycoplasma_genitalium__v__Mycoplasma_hyopneumoniae.tsv
└── Orthologues_Mycoplasma_hyopneumoniae
    ├── Mycoplasma_hyopneumoniae__v__Mycoplasma_agalactiae.tsv
 

We will start by finding the orthologs of a gene of interest. In the `Orthologues` directory, there is a separate subdirectory for each species.

Open `Orthologues/Orthologues_Mycoplasma_hyopneumoniae/Mycoplasma_hyopneumoniae__v__Mycoplasma_agalactiae.tsv` in a spreadsheet program, specifying that the file is tab-delimited if necessary. The file has three columns: `Orthogroup`, `Mycoplasma_hyopneumoniae`, and `Mycoplasma_agalactiae`.

Find `gi|71851854|gb|AAZ44462.1|` in the table. In this example, the gene belongs to orthogroup `OG0000014`, and its orthologs are `gi|290752976|emb|CBH40952.1|`, `gi|290752482|emb|CBH40454.1|`, and `gi|290752494|emb|CBH40466.1|`.


In [17]:
# df = pd.read_csv(f"{RESULTS_DIR_CORE}/Orthologues/Mycoplasma_agalactiae.tsv.gz", sep="\t", compression="gzip")
df = pd.read_csv(f"{RESULTS_DIR_CORE}/Orthologues/Orthologues_Mycoplasma_hyopneumoniae/Mycoplasma_hyopneumoniae__v__Mycoplasma_agalactiae.tsv", sep="\t")
with pd.option_context('display.max_colwidth', None):
    display(df.loc[df["Orthogroup"] == "OG0000014", :])

,Orthogroup,Mycoplasma_hyopneumoniae,Mycoplasma_agalactiae
9,OG0000014,gi|71851854|gb|AAZ44462.1|,"gi|290752482|emb|CBH40454.1|, gi|290752494|emb|CBH40466.1|, gi|290752976|emb|CBH40952.1|"


In [18]:
%%bash
RESULTS_DIR=$(ls -d OrthoFinder/ExampleData/OrthoFinder/Results_*/ | tail -n 1)
echo "Found results directory: $RESULTS_DIR"
ls -al "$RESULTS_DIR/Orthogroup_Sequences" | head -n 8
ls -al "$RESULTS_DIR/MultipleSequenceAlignments" | head -n 8

Found results directory: OrthoFinder/ExampleData/OrthoFinder/Results_Sep02/
total 4676
drwxr-sr-x  2 jovyan users 36864 Sep  2 16:02 .
drwxr-sr-x 15 jovyan users  4096 Sep  2 16:02 ..
-rw-r--r--  1 jovyan users 36442 Sep  2 16:02 OG0000000.fa
-rw-r--r--  1 jovyan users 18146 Sep  2 16:02 OG0000001.fa
-rw-r--r--  1 jovyan users  5643 Sep  2 16:02 OG0000002.fa
-rw-r--r--  1 jovyan users  8830 Sep  2 16:02 OG0000003.fa
-rw-r--r--  1 jovyan users 14113 Sep  2 16:02 OG0000004.fa
total 2724
drwxr-sr-x  2 jovyan users 20480 Sep  2 16:02 .
drwxr-sr-x 15 jovyan users  4096 Sep  2 16:02 ..
-rw-r--r--  1 jovyan users 48803 Sep  2 16:02 OG0000000.fa
-rw-r--r--  1 jovyan users 28968 Sep  2 16:02 OG0000001.fa
-rw-r--r--  1 jovyan users  6301 Sep  2 16:02 OG0000002.fa
-rw-r--r--  1 jovyan users 12224 Sep  2 16:02 OG0000003.fa
-rw-r--r--  1 jovyan users 32089 Sep  2 16:02 OG0000004.fa


<a id="sec-5-3-2"></a>
#### 5.3.2 Gene trees

Next, we will examine a gene tree to see how these orthologous relationships arose. OrthoFinder infers orthologs from resolved gene trees using a Duplication-Loss-Coalescence analysis to identify the most parsimonious interpretation of the tree. See the OrthoFinder 2 paper for more details.

All resolved gene trees are stored in `Resolved_Gene_Trees/Resolved_Gene_Trees.txt`. Each line contains an orthogroup ID, for example `OG0000008:`, followed by the corresponding gene tree. To find the tree for a particular orthogroup, search for its orthogroup ID.

Here, we will examine the tree for `OG0000008`.


In [19]:
%%bash
RESULTS_DIR=$(ls -d OrthoFinder/ExampleData/OrthoFinder/Results_*/ | tail -n 1)
echo "Found results directory: $RESULTS_DIR"
grep "^OG0000008:" "$RESULTS_DIR/Resolved_Gene_Trees/Resolved_Gene_Trees.txt"

Found results directory: OrthoFinder/ExampleData/OrthoFinder/Results_Sep02/
OG0000008: ((Mycoplasma_hyopneumoniae_gi|144227629|gb|AAZ44523.2|:1.36546,Mycoplasma_hyopneumoniae_gi|144227727|gb|AAZ44698.2|:0.70415)n1:0.460875,((Mycoplasma_gallisepticum_gi|284812289|gb|AAP57058.2|:0.34583,(Mycoplasma_gallisepticum_gi|284812274|gb|AAP57044.2|:0.13983,Mycoplasma_gallisepticum_gi|284812288|gb|AAP57057.2|:0.12558)n4:0.34393)n3:0.44562,(Mycoplasma_gallisepticum_gi|31541755|gb|AAP57054.1|:0.42184,(Mycoplasma_gallisepticum_gi|31541760|gb|AAP57059.1|:0.33692,(Mycoplasma_gallisepticum_gi|284812286|gb|AAP57055.2|:0.39052,(Mycoplasma_gallisepticum_gi|284812285|gb|ADB96899.1|:0.31571,(Mycoplasma_gallisepticum_gi|284812284|gb|ADB96898.1|:0.21708,Mycoplasma_gallisepticum_gi|284812290|gb|AAP57060.2|:0.09996)n9:0.08357)n8:0.13445)n7:0.10841)n6:0.16188)n5:0.65451)n2:0.460875)n0;


![image](https://raw.githubusercontent.com/OrthoFinder/OrthoFinder/gh-pages/assets/images/gene_tree.png)

<a id="sec-5-3-3"></a>
#### 5.3.3 Gene duplications

Because OrthoFinder infers gene trees, it can also identify gene duplication events. The `Gene_Duplication_Events` directory contains files that can be used to explore these events.

First, open `Gene_Duplication_Events/SpeciesTree_Gene_Duplications_0.5_Support.txt` in [iTOL](https://itol.embl.de/upload.cgi). In the iTOL control panel, open the **Advanced** tab and enable **Node IDs** to display the node labels.


In [20]:
%%bash
RESULTS_DIR=$(ls -d OrthoFinder/ExampleData/OrthoFinder/Results_*/ | tail -n 1)
echo "Found results directory: $RESULTS_DIR"
ls -al "$RESULTS_DIR/Gene_Duplication_Events"

Found results directory: OrthoFinder/ExampleData/OrthoFinder/Results_Sep02/
total 132
drwxr-sr-x  2 jovyan users   4096 Sep  2 16:02 .
drwxr-sr-x 15 jovyan users   4096 Sep  2 16:02 ..
-rw-r--r--  1 jovyan users 119432 Sep  2 16:02 Duplications.tsv
-rw-r--r--  1 jovyan users    174 Sep  2 16:02 SpeciesTree_Gene_Duplications_0.5_Support.txt


In [21]:
df = pd.read_csv(f"{RESULTS_DIR_CORE}/Gene_Duplication_Events/Duplications.tsv", sep="\t")
with pd.option_context('display.max_colwidth', None):
    display(df.tail(5))

,Orthogroup,Species Tree Node,Gene Tree Node,Support,Type,Genes 1,Genes 2
333,OG0000330,Mycoplasma_hyopneumoniae,n1,1.0,Terminal,Mycoplasma_hyopneumoniae_gi|144227586|gb|AAZ44428.2|,"Mycoplasma_hyopneumoniae_gi|71851803|gb|AAZ44411.1|, Mycoplasma_hyopneumoniae_gi|144227585|gb|AAZ44427.2|"
334,OG0000330,Mycoplasma_hyopneumoniae,n2,1.0,Terminal,Mycoplasma_hyopneumoniae_gi|71851803|gb|AAZ44411.1|,Mycoplasma_hyopneumoniae_gi|144227585|gb|AAZ44427.2|
335,OG0000331,Mycoplasma_hyopneumoniae,n0,1.0,Terminal,Mycoplasma_hyopneumoniae_gi|144227625|gb|AAZ44516.2|,"Mycoplasma_hyopneumoniae_gi|144227644|gb|AAZ44545.2|, Mycoplasma_hyopneumoniae_gi|71851902|gb|AAZ44510.1|, Mycoplasma_hyopneumoniae_gi|71851936|gb|AAZ44544.1|"
336,OG0000331,Mycoplasma_hyopneumoniae,n1,1.0,Terminal,Mycoplasma_hyopneumoniae_gi|144227644|gb|AAZ44545.2|,"Mycoplasma_hyopneumoniae_gi|71851902|gb|AAZ44510.1|, Mycoplasma_hyopneumoniae_gi|71851936|gb|AAZ44544.1|"
337,OG0000331,Mycoplasma_hyopneumoniae,n2,1.0,Terminal,Mycoplasma_hyopneumoniae_gi|71851902|gb|AAZ44510.1|,Mycoplasma_hyopneumoniae_gi|71851936|gb|AAZ44544.1|


In [22]:
%%bash
RESULTS_DIR=$(ls -d OrthoFinder/ExampleData/OrthoFinder/Results_*/ | tail -n 1)
echo "Found results directory: $RESULTS_DIR"
cat "$RESULTS_DIR/Gene_Duplication_Events/SpeciesTree_Gene_Duplications_0.5_Support.txt"

Found results directory: OrthoFinder/ExampleData/OrthoFinder/Results_Sep02/
((Mycoplasma_agalactiae_123:0.51407,Mycoplasma_hyopneumoniae_69:0.53945)N1_6:0.25523,(Mycoplasma_genitalium_10:0.53447,Mycoplasma_gallisepticum_119:0.4496)N2_11:0.25523)N0_0;

![image](https://raw.githubusercontent.com/OrthoFinder/OrthoFinder/gh-pages/assets/images/duplication_tree.png)

This tree summarizes gene duplication events. Each node label contains the node name followed by an underscore and the number of well-supported gene duplication events mapped to that node in the species tree.

Gene duplication events are considered well supported when at least 50% of the descendant species retain both copies of the duplicated gene. For node `N2`, there are six such well-supported duplication events. The numbers following the species names indicate the number of terminal duplications mapped to each species rather than to an internal node of the species tree.


In [23]:
%%bash
RESULTS_DIR=$(ls -d OrthoFinder/ExampleData/OrthoFinder/Results_*/ | tail -n 1)
echo "Found results directory: $RESULTS_DIR"
ls -al "$RESULTS_DIR"

Found results directory: OrthoFinder/ExampleData/OrthoFinder/Results_Sep02/
total 124
drwxr-sr-x 15 jovyan users  4096 Sep  2 16:02 .
drwxr-sr-x  3 jovyan users  4096 Sep  2 16:01 ..
-rw-r--r--  1 jovyan users  2955 Sep  2 16:02 Citation.txt
drwxr-sr-x  2 jovyan users  4096 Sep  2 16:02 Comparative_Genomics_Statistics
drwxr-sr-x  2 jovyan users  4096 Sep  2 16:02 Gene_Duplication_Events
-rw-r--r--  1 jovyan users   852 Sep  2 16:02 Log.txt
drwxr-sr-x  2 jovyan users 20480 Sep  2 16:02 MultipleSequenceAlignments
drwxr-sr-x  2 jovyan users  4096 Sep  2 16:02 Orthogroups
drwxr-sr-x  2 jovyan users 36864 Sep  2 16:02 Orthogroup_Sequences
drwxr-sr-x  6 jovyan users  4096 Sep  2 16:02 Orthologues
drwxr-sr-x  2 jovyan users  4096 Sep  2 16:02 Phylogenetically_Misplaced_Genes
drwxr-sr-x  2 jovyan users  4096 Sep  2 16:02 Phylogenetic_Hierarchical_Orthogroups
drwxr-sr-x  2 jovyan users  4096 Sep  2 16:02 Putative_Xenologs
drwxr-sr-x  2 jovyan users  4096 Sep  2 16:02 Resolved_Gene_Trees
drwxr-s

<a id="sec-6"></a>
## 6. Core/Assign

For large analyses, OrthoFinder can separate the workflow into a **core** analysis and a subsequent **assign** step. A representative set of species is first analyzed to establish the core orthogroups, gene trees, and species-tree framework. Additional species can then be assigned to this existing analysis without repeating the complete analysis from the beginning.

The example below uses the results generated from `ExampleData` as the core analysis and assigns the sequences in `ExampleData/AdditionalSpecies` to it. The `-pof` option is included so that pairwise ortholog files are also produced.


In [25]:
%%bash
cd OrthoFinder
RESULTS_DIR=$(ls -d ExampleData/OrthoFinder/Results_*/ | tail -n 1)
echo "Found results directory: $RESULTS_DIR"
../of3_env/bin/python -m orthofinder --core "$RESULTS_DIR" --assign ExampleData/AdditionalSpecies -pof

Found results directory: ExampleData/OrthoFinder/Results_Sep02/

2026-09-02 16:03:51 : Starting OrthoFinder v3.1.5.post1.dev8
32 thread(s) for highly parallel tasks (BLAST searches etc.)
4 thread(s) for OrthoFinder algorithm

INFO: For --assign defaulting to 'famsa' to reduce RAM usage

INFO: For --assign defaulting to 'FastTree -fastest' to reduce RAM usage

OrthoFinder version 3.1.5.post1.dev8 Copyright (C) 2014 David Emms

Results directory:
    /home/jovyan/6-OrthoFinder/OrthoFinder/ExampleData/OrthoFinder/Results_Sep02
_1/

Checking required programs are installed
Running with the recommended MSA tree inference by default. To revert to legacy 
method use "-M dendroblast".

Test can run "famsa" - ok
Test can run "fasttree_fastest" - ok
Test can run "astral-pro" - ok

Creating orthogroup profiles
----------------------------
Processing... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   1088/1088 0:00:008/1088 0:00:00

Adding new species in 
/home/jovyan/6-OrthoFinder/OrthoFinder/ExampleD

In [26]:
%%bash
RESULTS_DIR=$(ls -d OrthoFinder/ExampleData/OrthoFinder/Results_*/ | tail -n 1)
echo "Found results directory: $RESULTS_DIR"
ls -al "$RESULTS_DIR/" 

Found results directory: OrthoFinder/ExampleData/OrthoFinder/Results_Sep02_1/
total 116
drwxr-sr-x 15 jovyan users  4096 Sep  2 16:04 .
drwxr-sr-x  4 jovyan users  4096 Sep  2 16:03 ..
-rw-r--r--  1 jovyan users  2955 Sep  2 16:04 Citation.txt
drwxr-sr-x  2 jovyan users  4096 Sep  2 16:04 Comparative_Genomics_Statistics
drwxr-sr-x  2 jovyan users  4096 Sep  2 16:04 Gene_Duplication_Events
-rw-r--r--  1 jovyan users   792 Sep  2 16:04 Log.txt
drwxr-sr-x  2 jovyan users 20480 Sep  2 16:04 MultipleSequenceAlignments
drwxr-sr-x  2 jovyan users  4096 Sep  2 16:04 Orthogroups
drwxr-sr-x  2 jovyan users 36864 Sep  2 16:04 Orthogroup_Sequences
drwxr-sr-x  2 jovyan users  4096 Sep  2 16:04 Orthologues
drwxr-sr-x  2 jovyan users  4096 Sep  2 16:04 Phylogenetically_Misplaced_Genes
drwxr-sr-x  2 jovyan users  4096 Sep  2 16:04 Phylogenetic_Hierarchical_Orthogroups
drwxr-sr-x  2 jovyan users  4096 Sep  2 16:04 Putative_Xenologs
drwxr-sr-x  2 jovyan users  4096 Sep  2 16:04 Resolved_Gene_Trees
drwxr